# AED 3: Fluxo Transacional e Comportamento de PIX
**Vertical:** Transactional Analytics | **Tabela:** `public.transacoes`

**Objetivo Analítico:** 
Mapear a dinâmica de movimentação financeira do banco. Analisaremos o ticket médio por tipo de transação, a volumetria horária (identificando picos de uso) e a dominância do PIX frente a outros métodos, cruzando esses comportamentos com o perfil dos clientes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import os
from dotenv import load_dotenv

# Configurações de Ambiente (Docker Port 5433)
os.environ["PGCLIENTENCODING"] = "latin1"
load_dotenv(override=True)

db_url = URL.create(
    drivername="postgresql",
    username=os.getenv('POSTGRES_USER', 'caleb').strip(),
    password=os.getenv('POSTGRES_PASSWORD', '').strip(),
    host=os.getenv('POSTGRES_HOST', '127.0.0.1').strip(),
    port=os.getenv('POSTGRES_PORT', '5433').strip(),
    database=os.getenv('POSTGRES_DB', 'bank_twin').strip()
)

engine = create_engine(db_url)

# Configuração visual Dark Neon
sns.set_theme(style="darkgrid", rc={
    "axes.facecolor": "#111111", "figure.facecolor": "#111111", 
    "text.color": "white", "axes.labelcolor": "white", 
    "xtick.color": "white", "ytick.color": "white",
    "grid.color": "#333333"
})

# Carga dos Dados
df_transacoes = pd.read_sql("SELECT * FROM transacoes", engine)
df_clientes = pd.read_sql("SELECT cliente_id, segmento FROM clientes", engine)
df_contas = pd.read_sql("SELECT conta_id, cliente_id FROM contas", engine)

# Feature Engineering: Tempo e Sazonalidade
df_transacoes['data_transacao'] = pd.to_datetime(df_transacoes['data_transacao'])
df_transacoes['hora_dia'] = df_transacoes['data_transacao'].dt.hour
df_transacoes['dia_semana'] = df_transacoes['data_transacao'].dt.day_name()

print("✅ Dados transacionais carregados e enriquecidos!")

ProgrammingError: (psycopg2.errors.UndefinedTable) relation "transacoes" does not exist
LINE 1: SELECT * FROM transacoes
                      ^

[SQL: SELECT * FROM transacoes]
(Background on this error at: https://sqlalche.me/e/14/f405)

In [ ]:
print("="*50)
print("🔍 1. ESTRUTURA E QUALIDADE: TABELA TRANSAÇÕES")
print("="*50)
display(df_transacoes.info())

print("\n📊 ESTATÍSTICAS DESCRITIVAS: VALORES TRANSACIONADOS")
display(df_transacoes[['valor']].describe())

print("\n📊 DISTRIBUIÇÃO POR TIPO DE TRANSAÇÃO")
display(df_transacoes['tipo_transacao'].value_counts(normalize=True) * 100)